# L10d Lab: Evaluation of Bandit Problem Algorithms
In this lab, we'll continue our discussion [of multi-armed bandit problems](https://en.wikipedia.org/wiki/Multi-armed_bandit). We look at the performance of the explore-first, $\epsilon$-greedy, and UCB1 algorithms on a simple stochastic bandit problem, then implement Thompson sampling and use it to solve a binary Bernoulli bandit problem.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Run three bandit algorithms on the same problem:__ The explore-first, epsilon-greedy, and UCB1 algorithms each use the rewards seen so far to decide which arm to pull next. We'll run all three on the same bandit and count how often each one pulls each arm.
> * __Compare the regret of the three algorithms:__ Regret measures how much reward an algorithm gives up compared with always pulling the best arm. We'll compute the regret of each algorithm after every round and compare how the three curves grow.
> * __Implement Thompson sampling:__ Thompson sampling keeps a Beta distribution for the success probability of each arm and pulls the arm whose sample from that distribution is highest. We'll complete a Thompson sampling function and check that it learns which arm is best.

In [the lecture](../L10c/CHEME-5800-L10c-Lecture-StochasticMultiarmBandits-Fall-2026.ipynb), we introduced the stochastic multi-armed bandit problem, regret, and algorithms that balance exploration and exploitation. Here, we run these algorithms on bandits where we know the answer, so we can see how each one finds the best arm.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

Your Thompson sampling function goes in [src/Compute.jl](src/Compute.jl). We call [the `L10dThompson.my_thompson_sampling(...)` function](src/Compute.jl) with its module name, `L10dThompson`, in front, so that re-running the setup cell picks up your edits.

### The Arms
First, let's implement the `world(...)` function. This function takes the action $a$ from our agent and returns the reward associated with choosing this action, i.e., $r\sim\mathcal{D}_{a}$. We'll use [a Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) to model the rewards, so every reward is a number between 0 and 1. A Beta distribution has two parameters $(\alpha,\beta)$ which determine its shape, and its expectation is $\alpha/(\alpha+\beta)$.

We store the reward distribution of each arm in the `arms::Array{Beta{Float64},1}` array; the bandit algorithms see only the rewards:

In [ ]:
arms = [
    Beta(2.0, 2.0),   # arm 1
    Beta(10.0, 2.0),  # arm 2
    Beta(2.0, 10.0),  # arm 3
    Beta(6.0, 20.0),  # arm 4
    Beta(20.0, 6.0),  # arm 5
    Beta(20.0, 10.0), # arm 6
    Beta(10.0, 20.0)  # arm 7
]; # the reward distribution 𝒟ₐ of each arm

world = a -> rand(arms[a]); # a -> ... defines a function of a: world(a) draws one independent reward from the fixed distribution of arm a

Which arm has the highest expected reward? Only the world knows this answer, the bandit algorithms do not! Let's tabulate the parameters, mean reward, and standard deviation of each arm:

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per arm

    # compute the mean and the standard deviation of the reward of each arm -
    for a ∈ eachindex(arms)
        d = arms[a]; # the reward distribution of arm a
        (α, β) = params(d); # the two parameters of the Beta distribution
        push!(df, (action = a, alpha = α, beta = β,
            mean_reward = round(mean(d), digits = 3), sd_reward = round(std(d), digits = 3)));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ The winner is the arm with the highest expected reward, which is set by the parameters of its Beta distribution: arm 2.

Let's plot the reward distributions of arm 2 and another arm, arm 6. [The `pdf(...)` function](https://juliastats.org/Distributions.jl/stable/univariate/#Distributions.pdf-Tuple{UnivariateDistribution,%20Real}) evaluates the probability density of a distribution:

In [ ]:
let

    # initialize -
    x = range(0.0, 1.0, length = 501); # rewards between 0 and 1

    # plot the reward distributions of arm 2 and arm 6 -
    plot(x, pdf.(arms[2], x), lw = 2, label = "Arm 2: (α, β) = (10, 2)", framestyle = :box,
        xlabel = "Reward r (AU)", ylabel = "Probability density (AU)", legend = :topleft)
    plot!(x, pdf.(arms[6], x), lw = 2, label = "Arm 6: (α, β) = (20, 10)")
end

__What do we see?__ The rewards of arm 2 cluster around a higher mean than those of arm 6, but the two distributions overlap. A single pull of arm 6 sometimes beats a single pull of arm 2, so an algorithm must pull each arm several times before it can tell which one is better.

### Constants
Finally, let's set some constants we'll use in the subsequent tasks. See the comment beside the value for a description of what it is, its permissible values, etc.

In [ ]:
K = 7; # number of arms for the bandit
T = 5000; # number of rounds for each decision task
seed = 5800; # seed for the random number generator, so each game can be repeated

___

## Task 1: Run the Three Bandit Algorithms
In this task, we run the explore-first, $\epsilon$-greedy, and UCB1 algorithms on our `K`-arm bandit problem and see which arm each one pulls. We've created a custom type for each algorithm, which holds the number of arms `K`. We create an instance of the type using [the `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build), then pass it, the number of rounds `T`, and our `world(...)` function to [the `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.solve-Tuple{AbstractBanditAlgorithmModel}), which plays the game.

__Output__: With the `return_history = true` option, the call returns a `NamedTuple` with two fields: `actions`, the arm $a_t$ pulled in each round, and `rewards`, a $T\times K$ results matrix with one row per round and one column per arm. Row $t$ holds the reward $r_t$ in column $a_t$ and zeros everywhere else.

The algorithms and our `world(...)` function draw random numbers, so each game cell first sets the seed with [the `Random.seed!(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Random.seed!) and gives the same game every time we run it.

__Discuss:__ Before we run anything: if every pull cost real money or a day of laboratory time, which of the three strategies would you trust most, and why?

__Prediction:__ By the end of the game, which algorithm will have pulled the best arm, arm 2, most often: {explore-first, $\epsilon$-greedy, or UCB1}?

### Explore-First Algorithm
The explore-first algorithm pulls each arm $N$ times, then pulls the arm with the highest sample mean for the remaining rounds. The package sets $N$ with the exploration count from the lecture:
$$
N\approx(T/K)^{2/3}(\log T)^{1/3},
$$
rounded to the nearest integer, with at least 1 and at most $\lfloor T/K\rfloor$ pulls per arm, so every arm is pulled at least once and the exploration fits in $T$ rounds. Neither limit is active for our $T$ and $K$. Let's compute $N$ to see how long the exploration phase lasts:


In [ ]:
N = round(Int, (T/K)^(2/3)*cbrt(log(T))); # pulls per arm while exploring, rounded to an integer
N = clamp(N, 1, fld(T, K)); # at least 1, and at most ⌊T/K⌋ so exploring fits in T rounds
println("each arm is pulled N = ", N, " times, so exploring takes the first ", K*N, " rounds")

We create [an instance of the `MyExploreFirstAlgorithmModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyExploreFirstAlgorithmModel) and play the game. We keep the returned `NamedTuple` in `game_explore_first` and its reward matrix in the `results_explore_first::Array{Float64,2}` variable:

In [ ]:
game_explore_first = let

    # initialize -
    Random.seed!(seed); # set the seed, so this cell gives the same game every time
    m = build(MyExploreFirstAlgorithmModel, (
        K = K, # number of arms
    ));

    # play the game -
    game = solve(m, T = T, world = world, return_history = true); # actions and rewards

    game # return
end;
results_explore_first = game_explore_first.rewards; # the T × K reward matrix, for the regret in Task 2

### Epsilon-Greedy Algorithm
The epsilon-greedy algorithm first pulls every arm once. After that initialization, it pulls a random arm with probability $\epsilon_t$, and otherwise pulls the arm with the highest sample mean. The package uses the exploration probability from the lecture:
$$
\epsilon_t=\min\left\{1,\;\underbrace{\left(\frac{K\log t}{t}\right)^{1/3}}_{>1\text{ in early rounds}}\right\}.
$$
For our $K$ arms, $\epsilon_t=1$ until $t$ grows larger than $K\log t$, so the first rounds after initialization are all random pulls. After that, $\epsilon_t$ shrinks as the game goes on.

We create [an instance of the `MyEpsilonGreedyAlgorithmModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyEpsilonGreedyAlgorithmModel) and play the game. We keep the returned `NamedTuple` in `game_eps_greedy` and its reward matrix in the `results_eps_greedy::Array{Float64,2}` variable:


In [ ]:
game_eps_greedy = let

    # initialize -
    Random.seed!(seed); # set the seed, so this cell gives the same game every time
    m = build(MyEpsilonGreedyAlgorithmModel, (
        K = K, # number of arms
    ));

    # play the game -
    game = solve(m, T = T, world = world, return_history = true); # actions and rewards

    game # return
end;
results_eps_greedy = game_eps_greedy.rewards; # the T × K reward matrix, for the regret in Task 2

### UCB1 Algorithm
The UCB1 algorithm pulls each arm once. Then, in each round $t$, it pulls the arm with the highest upper confidence bound, the sample mean reward $\bar{\mu}(a)$ plus a confidence radius:
$$
\bar{\mu}(a)+\sqrt{\frac{2\log(t)}{N_{a}}},
$$
where $N_{a}$ is the number of times arm $a$ has been pulled so far. The radius is large for an arm we have pulled only a few times, so the algorithm is optimistic about the arms it knows least about.

We create [an instance of the `MyUCB1AlgorithmModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyUCB1AlgorithmModel) and play the game. We keep the returned `NamedTuple` in `game_UCB1` and its reward matrix in the `results_UCB1::Array{Float64,2}` variable:

In [ ]:
game_UCB1 = let

    # initialize -
    Random.seed!(seed); # set the seed, so this cell gives the same game every time
    m = build(MyUCB1AlgorithmModel, (
        K = K, # number of arms
    ));

    # play the game -
    game = solve(m, T = T, world = world, return_history = true); # actions and rewards

    game # return
end;
results_UCB1 = game_UCB1.rewards; # the T × K reward matrix, for the regret in Task 2

Let's look at the first ten rounds of the UCB1 game:

In [ ]:
results_UCB1[1:10, :] # rounds 1 to 10, all arms

__What do we see?__ Each row has one nonzero entry, the reward for the arm pulled in that round. In the first $K$ rounds, UCB1 pulls each arm once.

### Compare the Pull Counts
Let's count how many times each algorithm pulled each arm, using the `actions` field of each game:

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per arm

    # count the rounds in which each algorithm pulled each arm -
    for a ∈ 1:K
        push!(df, (
            arm = a,
            mean_reward = round(mean(arms[a]), digits = 3), # known only to us, the oracle
            explore_first = count(==(a), game_explore_first.actions), # ==(a) is a function that tests whether its input equals a
            eps_greedy = count(==(a), game_eps_greedy.actions),
            UCB1 = count(==(a), game_UCB1.actions),
        ));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ All three algorithms pulled arm 2, the best arm, most often. Explore-first pulled every arm $N$ times, then only arm 2. The $\epsilon$-greedy algorithm spread its other pulls about evenly over the six other arms. UCB1 pulled arm 2 the least of the three, and it spread its other pulls unevenly.

### Things to think about

__Question:__ Explore-first and $\epsilon$-greedy spread their other pulls about evenly over the six other arms, but UCB1 did not. Which arms did UCB1 favor, and why? Briefly explain.

___


## Task 2: Compute the Empirical Regret for Each Algorithm
In this task, we compute the regret of each algorithm in our seeded game and compare how the three curves grow.

__Regret__: Let $\mu^{\star}$ be the mean of the best arm, i.e., $\mu^{\star} = \max_{a\in\mathcal{A}}\mu(a)$. The regret $R(t)$ of an algorithm after $t$ rounds is defined as:
$$
\begin{align*}
R(t) = t\cdot\mu^{\star} - \sum_{s=1}^{t}r_{s}.
\end{align*}
$$
The first term is the expected reward from always choosing the best arm over the $t$ rounds. The second term is the total reward obtained by the agent over the $t$ rounds, where $r_{s}$ is the reward received at round $s$ for the action chosen by the agent.

We are the oracle, so we know the mean reward of every arm, including $\mu^{\star}$, the mean of arm 2. For each algorithm, we compute $R(t)$ after every round $t=1,2,\dots,T$ and store the curves in the `regret_curves::Dict{String, Array{Float64,1}}` dictionary:

In [ ]:
regret_curves = let

    # initialize -
    μstar = maximum(mean.(arms)); # μ⋆, the mean reward of the best arm: known only to us, the oracle
    games = [("Explore-first", results_explore_first), ("ε-greedy", results_eps_greedy), ("UCB1", results_UCB1)]; # (name, results array)
    regret_curves = Dict{String, Array{Float64,1}}(); # the regret after each round, for each algorithm

    # main loop -
    for (name, U) ∈ games
        r = vec(sum(U, dims = 2)); # the reward in each round: each row of U has at most one nonzero entry
        R = zeros(T); # the regret after each round
        total = 0.0; # the reward collected so far
        for t ∈ 1:T
            total = total + r[t]; # r₁ + r₂ + ... + rₜ
            R[t] = t*μstar - total; # R(t) = tμ⋆ - (r₁ + r₂ + ... + rₜ)
        end
        regret_curves[name] = R; # store the curve for this algorithm
    end

    regret_curves # return
end;

Let's plot the regret curves.

__Prediction:__ Which algorithm will have the smallest regret at the end of the game: {explore-first, $\epsilon$-greedy, or UCB1}?

In [ ]:
let

    # initialize -
    labels = ["Explore-first", "ε-greedy", "UCB1"]; # plot the curves in this order
    p = plot(framestyle = :box, xlabel = "Round t", ylabel = "Regret R(t) (AU)", legend = :topleft);

    # plot the regret after each round -
    for name ∈ labels
        plot!(p, 1:T, regret_curves[name], lw = 2, label = name);
    end

    p # return
end

__What do we see?__ Explore-first's regret climbs while it pulls every arm in turn, then levels off once it commits to arm 2. The $\epsilon$-greedy regret keeps climbing, because it never stops pulling random arms, and it eventually passes explore-first's. UCB1's regret grows more and more slowly and ends the smallest. Each curve wiggles because the rewards are random.

### Things to think about

__Question:__ In our seeded game, UCB1 pulled the best arm less often than $\epsilon$-greedy, yet it had the smallest regret. How can that be? Briefly explain.

___


## Task 3: Implement Thompson Sampling
In this task, we implement Thompson sampling and use it on a binary Bernoulli bandit, where every reward is a success, $r=1$, or a failure, $r=0$.

Each arm $a\in\mathcal{A}$ succeeds with a probability $\theta_{a}$ the agent does not know, though we are [the Oracle](https://en.wikipedia.org/wiki/The_Oracle_(The_Matrix)) and do. Can the agent learn which arm is most likely to succeed? Let's choose the success probability of each of three arms:

In [ ]:
θ = [0.90, 0.60, 0.80]; # the probability of success of each arm: only the world knows these

### Complete the Thompson Sampling Function
The agent models its uncertainty about each $\theta_{a}$ with a Beta distribution. Before any pulls, its belief about arm $a$, the __prior__, is $\text{Beta}(\alpha(a),\beta(a))$ with $\alpha(a)=\beta(a)=1$, which is uniform on $[0,1]$. After $S(a)$ successes and $F(a)$ failures, its __posterior__ is $\text{Beta}(\alpha(a)+S(a),\beta(a)+F(a))$.

__Discuss:__ Our arms succeed or fail, like a reaction that does or does not reach its target yield. The uniform prior says we know nothing about an arm before we pull it. How might a chemist with years of experience choose a different prior, and what would that change?

Open [`src/Compute.jl`](src/Compute.jl) and complete [the `my_thompson_sampling(...)` function](src/Compute.jl), following the lecture's algorithm. A TODO comment marks each of the four lines you write:

> __Implementation tasks__
>
> * __TODO 1:__ Replace `0.0` in `p[a] = 0.0` with one sample from $\text{Beta}(\alpha(a)+S(a),\beta(a)+F(a))$, drawn with the random number generator `rng`.
> * __TODO 2:__ Replace `1` in `aₜ = 1` with the arm $a$ that maximizes the sample $p(a)$.
> * __TODO 3:__ Replace the right-hand sides of the two count lines so that $S(a_{t})\gets S(a_{t})+r_{t}$ and $F(a_{t})\gets F(a_{t})+(1-r_{t})$.

We play $T$ rounds against [the `bernoulli_world(...)` function](docs/compare_bandits.md#bernoulli-world), which returns `1` with probability $\theta_{a}$ and `0` otherwise, using [a Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution). The agent and the world share one generator, built from our `seed` by [the `MersenneTwister(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Random.MersenneTwister). The returned `result::NamedTuple` holds the `actions` and `rewards` of each round, and the counts `S` and `F`.

Save your edits, re-run the setup cell to reload them, then run the cell below. The untouched function throws an error, because it never updates the counts:

In [ ]:
result = let

    # initialize -
    rng = MersenneTwister(seed); # one random number generator for the agent and the world
    world_function = a -> bernoulli_world(a, θ, rng); # the agent sees only the reward of the arm it pulls

    # play T rounds with Thompson sampling -
    L10dThompson.my_thompson_sampling(world_function, length(θ), T, rng = rng) # the untouched stub throws an error
end;
println("pulls of each arm: ", result.S .+ result.F)

### Check: Your Thompson Sampling

Every round adds one success or one failure to the arm we pulled, so the counts must agree with the actions and rewards, and the same seed must give the same game. We also play a test bandit whose best arm is the last one, which Thompson sampling should pull in most rounds:

In [ ]:
@testset "My Thompson sampling" begin

    # every round adds one success or one failure to the arm we pulled -
    @test sum(result.S) + sum(result.F) == T
    @test result.S .+ result.F == [count(result.actions .== a) for a ∈ eachindex(θ)] # pulls of each arm
    @test result.S == [sum(result.rewards[result.actions .== a]) for a ∈ eachindex(θ)] # successes on each arm

    # the same seed gives the same game, every time we play it -
    for k ∈ 1:2
        rng = MersenneTwister(seed); # a new random number generator with the same seed
        again = L10dThompson.my_thompson_sampling(a -> bernoulli_world(a, θ, rng), length(θ), T, rng = rng);
        @test again.actions == result.actions
    end

    # a test bandit whose best arm is the last one -
    θ_test = [0.70, 0.50, 0.90]; # success probabilities of the test bandit
    rng_test = MersenneTwister(1234); # its own random number generator
    test = L10dThompson.my_thompson_sampling(a -> bernoulli_world(a, θ_test, rng_test), 3, 2000, rng = rng_test);
    @test count(test.actions .== 3) > 1000 # the best arm is pulled in most of the 2000 rounds
    @test abs((1 + test.S[3])/(2 + test.S[3] + test.F[3]) - 0.90) < 0.05 # its posterior mean is close to 0.9
end;

### What Did the Agent Learn?
Let's compare the agent's belief with the truth only we know. Its estimate of $\theta_{a}$ is the posterior mean:
$$
\hat{\theta}_{a} = \frac{\alpha(a)+S(a)}{\alpha(a)+\beta(a)+S(a)+F(a)}.
$$

__Prediction:__ For which arm will the estimate $\hat{\theta}_{a}$ be closest to the true value: {arm 1, arm 2, or arm 3}?

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per arm
    α = 1.0; # prior parameter α(a), the same for every arm
    β = 1.0; # prior parameter β(a), the same for every arm

    # compare the posterior mean with the true success probability -
    for a ∈ eachindex(θ)
        S = result.S[a]; # successes on arm a
        F = result.F[a]; # failures on arm a
        push!(df, (arm = a, θ = θ[a], pulls = S + F, S = S, F = F, θ̂ = round((α + S)/(α + β + S + F), digits = 3)));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ The agent pulled arm 1, the best arm, in almost every round, and its estimate of $\theta_{1}$ is close to the true value. It pulled arms 2 and 3 only rarely, so those estimates rest on far fewer pulls.

Let's plot the posterior of each arm with [the `pdf(...)` function](https://juliastats.org/Distributions.jl/stable/univariate/#Distributions.pdf-Tuple{UnivariateDistribution,%20Real}):

In [ ]:
let

    # initialize -
    x = range(0.0, 1.0, length = 2001); # success probabilities between 0 and 1, on a fine grid: the posterior of arm 1 is narrow
    p = plot(framestyle = :box, xlabel = "Probability of success (AU)", ylabel = "Probability density (AU)", legend = :topleft);

    # plot the posterior of each arm -
    for a ∈ eachindex(θ)
        d = Beta(1.0 + result.S[a], 1.0 + result.F[a]); # the posterior of arm a, with α(a) = β(a) = 1
        plot!(p, x, pdf.(d, x), lw = 2, label = "Arm $(a): θ = $(θ[a])");
    end

    p # return
end

__What do we see?__ The posterior of arm 1 is a tall, narrow peak near its true $\theta_{1}$: after many pulls, the agent is nearly sure of it. Arm 3's posterior is wider, and arm 2's is broad.

__Prediction:__ Suppose arm 3 is almost as good as arm 1. Will Thompson sampling pull arm 3 {more often, less often, or about as often} as before?

In [ ]:
let

    # initialize -
    θ_close = [0.90, 0.60, 0.88]; # arm 3 is now almost as good as arm 1
    rng = MersenneTwister(seed); # the same seed as before
    world_function = a -> bernoulli_world(a, θ_close, rng); # the agent sees only the reward of the arm it pulls

    # play T rounds with Thompson sampling -
    close_result = L10dThompson.my_thompson_sampling(world_function, length(θ_close), T, rng = rng);
    println("θ = ", θ_close, ": pulls of each arm ", close_result.S .+ close_result.F)
    println("posterior means: ", round.((1 .+ close_result.S) ./ (2 .+ close_result.S .+ close_result.F), digits = 3))
end

__What do we see?__ When arm 3 is almost as good as arm 1, the agent pulls arm 3 much more often. It takes more pulls to tell two nearly equal arms apart.

### Things to think about

__Question:__ The agent pulled arm 2 only a handful of times, so its estimate of $\theta_{2}$ is still rough. Is that a problem for the agent, and why does Thompson sampling stop pulling an arm before it knows the arm well? Briefly explain.

___

## Summary
In this lab, we ran the explore-first, $\epsilon$-greedy, and UCB1 algorithms on the same bandit, measured and compared their regret over the rounds of one game, and implemented Thompson sampling for a binary Bernoulli bandit.

> __Key Takeaways:__
>
> * __Three ways to balance exploration and exploitation:__ All three algorithms found the best arm, but they explored in different ways. Explore-first and epsilon-greedy spread their exploration evenly over the arms, while UCB1 spent most of its exploration on the arms whose mean rewards were close to the best.
>
> * __Regret depends on which arms we explore:__ UCB1 pulled the best arm less often than epsilon-greedy but had the smallest regret, because most of its other pulls went to arms that were almost as good. Its regret also grew more and more slowly over the rounds, while epsilon-greedy's kept climbing, because epsilon-greedy never stops pulling random arms.
>
> * __Thompson sampling keeps a probability model of each arm:__ We completed a Thompson sampling function that samples from the posterior of each arm and updates the success and failure counts after each pull. It pulled the best arm in almost every round, and it pulled the worse arms only rarely, even before it knew their success probabilities well.

We can now change the arms, the number of rounds, or the seed and see how the pulls and the regret of each algorithm respond.

___